# Imports

In [118]:
from PyPDF2 import PdfReader
from langchain_groq import ChatGroq 
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

### API

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# Preprocessing

In [120]:
pdfreader = PdfReader('Foreign-Policy.pdf')

In [121]:
# ham pdf k har page se read krke raw_text mein daal rhe 
raw_text = ''

for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [122]:
raw_text

'PUBLIC POLICY IN-DEPTH\n®FOREIGN POLICYA Balance of Interests. At times, a national interest is easily categorized as economic, ideological, or security-related, but\nat other times, national interests defy categorization. For example, some Americans point to the Iraq War of 2003–\n2011 as an intersection of economic, ideological, and security interests. At the time of the invasion, the United States\nhad economic concerns about the effect of regional instability on the global oil supply, ideological concerns about the\nbrutal dictatorship of Saddam Hussein, and security concerns about whether or not the Hussein regime was stockpiling\nweapons.\nSometimes, a nation’s economic, ideological, and security interests simply do not align. For example, the United States\nhas an economic interest in trading with China, but Chinese human rights violations run counter to U.S. commitments\nto individual liberty and due process. The United States also has a security interest in supporting the gov

In [123]:
textsplitter = CharacterTextSplitter(
    separator= "\n",
    chunk_size = 800,
    chunk_overlap = 200,
    length_function = len
)

text = textsplitter.split_text(raw_text)

In [124]:
len(text)

5

In [125]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3237.05it/s]


In [126]:
vectorstore = FAISS.from_texts(text, embedding=embeddings)

# LLM

In [ ]:
llm = ChatGroq(
        temperature = 0.8,
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        max_tokens=1024,
    )

In [128]:
# provide the semantic 3 characters related to query from the vectorstore
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

# Q&A

In [140]:
query = 'what are economic interests'
relevant_chunks = vectorstore.similarity_search(query)

In [141]:
context = "\n\n".join([chunks.page_content for chunks in relevant_chunks])

In [142]:
prompt = f"""Use the context below to answer the question.
If answer is not in context, say "I don't know".

Context:
{context}

Question: {query}
"""

In [143]:
response = llm.invoke(prompt)

In [144]:
print(response.content)

According to the context, economic interests include:

1. Providing citizens with an adequate standard of living
2. Ensuring economic development and growth
3. Establishing trade relations with other nations
4. Protecting economic investment abroad and at home
5. Protecting the means and routes of trade
6. Protecting the competitiveness of key domestic industries
7. Maintaining economic power to ensure economic self-determination.
